In [1]:
# MCS underflow/overflow check for exiting muons -- reviewer question (London)
#
# SPINE's mcs_fit (spine/utils/mcs.py) bounds the MLE kinetic-energy search to
# lower_bound=10.0 MeV, upper_bound=100000.0 MeV (100 GeV) by default. This
# script checks, on the full nominal MC sample, what fraction of exiting
# (uncontained) muons in the selected sample pile up at or near these bounds.

import uproot
import numpy as np

DATA_FILE = (
    "/Users/rvizarreta/Library/CloudStorage/GoogleDrive-rvizarreta14@gmail.com/"
    "My Drive/\U0001F3DB PhD Repository/\U0001F680 Research/\U0001F916 Experiments&Projects/"
    "ICARUS/ICARUS_CC0pi_Selection/data/icarus_numi_numu_mc_onbeam_offbeam_syst_ppfx.root"
)

LOWER_BOUND = 10.0       # MeV, SPINE mcs_fit default lower_bound
UPPER_BOUND = 100000.0   # MeV, SPINE mcs_fit default upper_bound

f = uproot.open(DATA_FILE)
t = f["events/full/selected"]

arr = t.arrays(
    ["reco_leading_muon_containment", "reco_leading_muon_mcs_ke"],
    library="np",
)
containment = arr["reco_leading_muon_containment"]
mcs_ke = arr["reco_leading_muon_mcs_ke"]

exiting_mask = containment == 0
exiting_ke = mcs_ke[exiting_mask]
n_exiting = len(exiting_ke)

print(f"Total selected entries: {len(mcs_ke)}")
print(f"Exiting (uncontained) muons: {n_exiting}")
print()

print("MCS KE distribution (exiting muons):")
print(f"  min = {np.nanmin(exiting_ke):.3f} MeV")
print(f"  max = {np.nanmax(exiting_ke):.3f} MeV")
pct = np.nanpercentile(exiting_ke, [0, 1, 5, 50, 95, 99, 100])
print(f"  percentiles [0,1,5,50,95,99,100] = {pct}")
print()

# Exact pile-up at the MLE bounds
n_at_lower = np.sum(exiting_ke == LOWER_BOUND)
n_at_upper = np.sum(exiting_ke == UPPER_BOUND)
print(f"Exactly at lower bound ({LOWER_BOUND} MeV): "
      f"{n_at_lower} ({100 * n_at_lower / n_exiting:.3f}%)")
print(f"Exactly at upper bound ({UPPER_BOUND} MeV): "
      f"{n_at_upper} ({100 * n_at_upper / n_exiting:.3f}%)")
print()

# Check whether the pile-up is a hard spike at the boundary or a smooth
# approach -- compare counts at a few thresholds below the upper bound.
for thresh in [1000, 5000, 10000, 50000, 90000, 99999]:
    n_above = np.sum(exiting_ke >= thresh)
    print(f"  >= {thresh:>6} MeV: {n_above:6d} ({100 * n_above / n_exiting:.3f}%)")

n_between = np.sum((exiting_ke >= 90000) & (exiting_ke < UPPER_BOUND))
print(f"\nEvents in [90000, {UPPER_BOUND}) MeV, i.e. near but not at the "
      f"upper bound: {n_between}")

# For reference: same check on contained muons (MCS is computed for all
# tracks by default in SPINE's MCSEnergyProcessor, even though the analysis
# uses range-based momentum for contained muons instead of MCS).
contained_ke = mcs_ke[~exiting_mask]
n_contained = len(contained_ke)
n_at_upper_contained = np.sum(contained_ke == UPPER_BOUND)
print(f"\n[Reference only, not used in the analysis] Contained muons at "
      f"upper bound: {n_at_upper_contained} of {n_contained} "
      f"({100 * n_at_upper_contained / n_contained:.3f}%)")

Total selected entries: 98983
Exiting (uncontained) muons: 69119

MCS KE distribution (exiting muons):
  min = 143.452 MeV
  max = 100000.000 MeV
  percentiles [0,1,5,50,95,99,100] = [   143.45193481    183.23565063    277.86413574    927.85467529
   3737.00917969 100000.         100000.        ]

Exactly at lower bound (10.0 MeV): 0 (0.000%)
Exactly at upper bound (100000.0 MeV): 2205 (3.190%)

  >=   1000 MeV:  31342 (45.345%)
  >=   5000 MeV:   2843 (4.113%)
  >=  10000 MeV:   2342 (3.388%)
  >=  50000 MeV:   2210 (3.197%)
  >=  90000 MeV:   2206 (3.192%)
  >=  99999 MeV:   2205 (3.190%)

Events in [90000, 100000.0) MeV, i.e. near but not at the upper bound: 1

[Reference only, not used in the analysis] Contained muons at upper bound: 3 of 29864 (0.010%)
